# PMT terminal noise and 900 V dark-response comparison

This self-contained notebook compares oscilloscope terminals 2, 3, and 4 under three empty-input conditions, then compares the powered PMT dark runs on terminals 3 and 4 with the 900 V dataset used by `PMT_PROCESSING.ipynb`.

The analyses are deliberately separated:

- **Empty input:** baseline RMS, offsets, excursions, average waveform, and PSD.
- **PMT at 900 V:** triggered dark-event rate, pulse height, pulse charge, and average anode current.

Normal-trigger and powered-PMT records are trigger-selected populations; they are not unbiased samples of ordinary baseline noise.

## 1. Imports and configuration

All executable analysis is contained in this notebook. Large LeCroy exports are streamed one file at a time; every waveform contributes to summary statistics, while only a limited number are retained for waveform and PSD plots.

## 2. LeCroy loading and empty-input statistics

In [ ]:
# Locate the repository when Jupyter starts in Notebooks/.
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks': PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import SRC.KingCRAB.pmt_noise as pmt_noise_helpers
from SRC.KingCRAB.context import configure_module
from SRC.KingCRAB.pmt_noise import baseline_mask, common_stack, dark_file_observables, median_psd, process_dark_run, process_empty_dataset, read_waveforms, trapezoid_integral, trigger_elapsed_time, waveform_statistics

configure_module(pmt_noise_helpers, globals())

## 3. Process the nine empty-input combinations

`No SHV/2` remains in the table even if empty, making missing data visible.

In [ ]:
empty_specs=[]
for terminal in TERMINALS:
    empty_specs += [
        (terminal,"Auto trigger","auto",BASE_FOLDER/"Auto Trigger"/f"No PMT {terminal}",f"C{terminal}C1*.txt"),
        (terminal,"Normal trigger","negative",BASE_FOLDER/"Normal Trigger"/f"No PMT {terminal}",f"C{terminal}C1*.txt"),
        (terminal,"No SHV","auto",BASE_FOLDER/"No SHV"/str(terminal),f"C{terminal}C1*.txt"),
    ]

empty_results=[]; empty_summary=[]
for terminal,condition,mode,folder,pattern in empty_specs:
    files,rows,corrected=process_empty_dataset(folder,pattern,mode)
    rms=np.array([r["rms_V"] for r in rows]); base=np.array([r["baseline_V"] for r in rows]); p2p=np.array([r["peak_to_peak_V"] for r in rows])
    n=len(rows)
    summary={
        "condition":condition,"terminal":terminal,"files":len(files),"waveforms":n,
        "baseline_median_mV":np.median(base)*1e3 if n else np.nan,
        "rms_median_mV":np.median(rms)*1e3 if n else np.nan,
        "rms_p16_mV":np.percentile(rms,16)*1e3 if n else np.nan,
        "rms_p84_mV":np.percentile(rms,84)*1e3 if n else np.nan,
        "peak_to_peak_median_mV":np.median(p2p)*1e3 if n else np.nan,
        "absolute_crossing_fraction":np.mean([r["absolute_threshold_crossing"] for r in rows]) if n else np.nan,
        "normalized_crossing_fraction":np.mean([r["normalized_threshold_crossing"] for r in rows]) if n else np.nan,
    }
    empty_summary.append(summary)
    empty_results.append({"terminal":terminal,"condition":condition,"mode":mode,"rows":rows,"corrected":corrected})
    print(f"Terminal {terminal}, {condition:14s}: {len(files):3d} files, {n:6d} waveforms")

empty_table=pd.DataFrame(empty_summary)
display(empty_table.round(5))

## 4. Empty-input comparison plots

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.8))
pos=np.arange(len(CONDITIONS)); offsets={2:-.22,3:0,4:.22}
for terminal in TERMINALS:
    d=empty_table[(empty_table.terminal==terminal)&(empty_table.waveforms>0)].set_index("condition")
    x=[]; y=[]; lo=[]; hi=[]; p2p=[]
    for j,c in enumerate(CONDITIONS):
        if c not in d.index: continue
        r=d.loc[c]; x.append(j+offsets[terminal]); y.append(r.rms_median_mV)
        lo.append(r.rms_median_mV-r.rms_p16_mV); hi.append(r.rms_p84_mV-r.rms_median_mV); p2p.append(r.peak_to_peak_median_mV)
    axes[0].errorbar(x,y,yerr=[lo,hi],marker='o',capsize=3,color=COLORS[terminal],label=f"Terminal {terminal}")
    axes[1].plot(x,p2p,'o-',color=COLORS[terminal],label=f"Terminal {terminal}")
for ax in axes:
    ax.set_xticks(pos,CONDITIONS); ax.grid(alpha=.25,axis='y'); ax.legend()
axes[0].set(ylabel="Median RMS [mV]",title="Baseline noise (16–84% interval)")
axes[1].set(ylabel="Median peak-to-peak [mV]",title="Full-trace excursions")

fig,axes=plt.subplots(1,3,figsize=(15,4.5),sharey=True)
for ax,condition in zip(axes,CONDITIONS):
    for result in empty_results:
        if result["condition"]!=condition: continue
        vals=np.array([r["rms_V"] for r in result["rows"]])*1e3
        if len(vals): ax.hist(vals,bins=45,histtype='step',density=True,lw=1.7,color=COLORS[result['terminal']],label=f"Terminal {result['terminal']}")
    ax.set(title=condition,xlabel="Baseline RMS [mV]"); ax.grid(alpha=.25); ax.legend()
axes[0].set_ylabel("Probability density")

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4.5),sharey=True)
for ax,condition in zip(axes,CONDITIONS):
    for result in empty_results:
        if result["condition"]!=condition: continue
        vals=np.array([r["baseline_V"] for r in result["rows"]])*1e3
        if len(vals): ax.plot(vals,lw=.8,color=COLORS[result['terminal']],label=f"Terminal {result['terminal']}")
    ax.set(title=condition,xlabel="Waveform index"); ax.grid(alpha=.25); ax.legend()
axes[0].set_ylabel("Baseline [mV]")

for condition in CONDITIONS:
    selected=[r for r in empty_results if r['condition']==condition and r['corrected']]
    fig,axes=plt.subplots(1,2,figsize=(13,4.5))
    for result in selected:
        t,stack=common_stack(result['corrected'])
        axes[0].plot(t*1e9,np.mean(stack,axis=0)*1e3,color=COLORS[result['terminal']],label=f"Terminal {result['terminal']}")
        f,p=median_psd(result['corrected']); keep=f>0
        axes[1].loglog(f[keep],p[keep],color=COLORS[result['terminal']],label=f"Terminal {result['terminal']}")
    axes[0].axvline(0,color='k',ls='--',lw=1); axes[0].set(xlabel='Time [ns]',ylabel='Mean corrected voltage [mV]',title=f'{condition}: average waveform')
    axes[1].set(xlabel='Frequency [Hz]',ylabel='Median PSD [V²/Hz]',title=f'{condition}: spectrum')
    for ax in axes: ax.grid(alpha=.25,which='both'); ax.legend()

## 5. Powered-PMT dark-run functions

The LeCroy header contains one trigger timestamp per segment. Live time is estimated by summing each file's first-to-last trigger span; gaps while files are saved are excluded. This is much more appropriate for triggered acquisitions than summing the 200 ns displayed waveform windows.

Two rates are reported:

- **Hardware-trigger rate:** every stored trigger divided by live time.
- **Software-selected rate:** events whose negative pulse height exceeds three times that event's pre-trigger RMS.

These remain sensitive to the oscilloscope trigger level, dead time, PMT temperature, and light tightness.

In [ ]:
HEADER_RE=re.compile(r"^#\d+,(\d{2}-[A-Za-z]{3}-\d{4}) (\d{2}:\d{2}:\d{2}),(.*)$")
configure_module(pmt_noise_helpers, globals())

## 6. Analyze terminal 3 and terminal 4 at 900 V

In [ ]:
dark_results=[]; dark_summaries=[]
for dataset,cfg in DARK_CONFIG.items():
    terminal=cfg['terminal']
    files,frame,traces,live=process_dark_run(terminal,cfg['folder'],cfg['pattern'])
    n=len(frame); selected=frame.software_selected.sum() if n else 0
    q_selected=frame.loc[frame.software_selected,'charge_C'] if n else pd.Series(dtype=float)
    hardware_rate=n/live if live>0 else np.nan
    selected_rate=selected/live if live>0 else np.nan
    # Signed mean pulse charge avoids the positive-only noise bias of clipping.
    mean_q=q_selected.mean() if selected else np.nan
    anode_current=selected_rate*mean_q if selected else np.nan
    summary={
        'dataset':dataset,'terminal':terminal,'voltage_V':DARK_VOLTAGE,
        'files':len(files),'triggers':n,'live_time_s':live,
        'hardware_trigger_rate_Hz':hardware_rate,
        'software_selected':int(selected),'software_selected_fraction':selected/n if n else np.nan,
        'software_selected_rate_Hz':selected_rate,
        'median_baseline_rms_mV':frame.rms_V.median()*1e3 if n else np.nan,
        'baseline_rms_p16_mV':frame.rms_V.quantile(.16)*1e3 if n else np.nan,
        'baseline_rms_p84_mV':frame.rms_V.quantile(.84)*1e3 if n else np.nan,
        'median_pulse_height_mV':frame.height_V.median()*1e3 if n else np.nan,
        'median_selected_charge_pC':q_selected.median()*1e12 if selected else np.nan,
        'mean_selected_charge_pC':mean_q*1e12 if selected else np.nan,
        'inferred_anode_current_nA':anode_current*1e9 if selected else np.nan,
    }
    dark_summaries.append(summary)
    dark_results.append({'dataset':dataset,'terminal':terminal,'files':files,'frame':frame,'traces':traces})
    safe_name=dataset.lower().replace(' ','_').replace(',','')

dark_table=pd.DataFrame(dark_summaries)
display(dark_table.round(5))

## 7. Direct noise and dark-current comparison

All three powered datasets are processed with the same pre-trigger baseline window, 3σ pulse-height selection, timestamp-derived live time, and signed charge integration. This avoids the incompatible live-time calculation previously present in `PMT_PROCESSING.ipynb`.

The left panel compares powered-run baseline RMS and shows the matching empty-terminal auto-trigger RMS as dashed references. The right panel compares inferred anode dark current, calculated as selected-event rate times mean selected charge.

In [ ]:
labels=list(dark_table.dataset)
x=np.arange(len(labels))
fig,axes=plt.subplots(1,2,figsize=(14,5))
noise=dark_table.median_baseline_rms_mV.to_numpy()
noise_low=noise-dark_table.baseline_rms_p16_mV.to_numpy()
noise_high=dark_table.baseline_rms_p84_mV.to_numpy()-noise
bar_colors=[DARK_COLORS[label] for label in labels]
axes[0].errorbar(x,noise,yerr=[noise_low,noise_high],fmt='none',ecolor='black',capsize=4,zorder=3)
axes[0].bar(x,noise,color=bar_colors,alpha=.85)
for terminal in (3,4):
    ref=empty_table[(empty_table.condition=='Auto trigger')&(empty_table.terminal==terminal)]
    if len(ref): axes[0].axhline(ref.iloc[0].rms_median_mV,color=COLORS[terminal],ls='--',lw=1.5,label=f'Empty terminal {terminal}: {ref.iloc[0].rms_median_mV:.3f} mV')
axes[0].set(ylabel='Median pre-trigger RMS [mV]',title='Powered-run noise versus empty-terminal noise')
axes[0].legend()

current=dark_table.inferred_anode_current_nA.to_numpy()
axes[1].bar(x,current,color=bar_colors,alpha=.85)
axes[1].set(ylabel='Inferred anode dark current [nA]',title='900 V inferred dark current')
for i,value in enumerate(current): axes[1].text(i,value,f'{value:.4g}',ha='center',va='bottom',fontsize=9)
for ax in axes:
    ax.set_xticks(x,labels,rotation=18,ha='right'); ax.grid(alpha=.25,axis='y')

fig,axes=plt.subplots(1,3,figsize=(16,4.8))
height_pool=np.concatenate([r['frame'].height_V.to_numpy()*1e3 for r in dark_results])
charge_pool=np.concatenate([r['frame'].loc[r['frame'].software_selected,'charge_C'].to_numpy()*1e12 for r in dark_results])
height_range=(0,np.nanpercentile(height_pool,99.5)); charge_range=tuple(np.nanpercentile(charge_pool,[0.5,99.5]))
for result in dark_results:
    label=result['dataset']; frame=result['frame']; selected=frame[frame.software_selected]; color=DARK_COLORS[label]
    axes[0].hist(frame.height_V*1e3,bins=100,range=height_range,histtype='step',density=True,lw=1.6,color=color,label=label)
    axes[1].hist(selected.charge_C*1e12,bins=100,range=charge_range,histtype='step',density=True,lw=1.6,color=color,label=label)
for i,r in dark_table.iterrows():
    err=np.sqrt(r.software_selected)/r.live_time_s if r.live_time_s>0 else np.nan
    axes[2].errorbar(i,r.software_selected_rate_Hz,yerr=err,fmt='o',capsize=4,ms=8,color=DARK_COLORS[r.dataset])
axes[0].set(xlabel='Triggered pulse height [mV]',ylabel='Probability density',title='All triggers (0–99.5th percentile)')
axes[1].set(xlabel='Selected pulse charge [pC]',ylabel='Probability density',title=f'>{DARK_SOFTWARE_THRESHOLD_SIGMA:.0f}σ events (0.5–99.5th percentile)')
axes[2].set(xlabel='Powered dataset',ylabel='Selected dark-event rate [Hz]',title='900 V dark-event rate')
axes[2].set_xticks(x,labels,rotation=18,ha='right')
for ax in axes: ax.grid(alpha=.25); 
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)

fig,axes=plt.subplots(1,2,figsize=(13,4.7))
for result in dark_results:
    label=result['dataset']; t,stack=common_stack(result['traces']); color=DARK_COLORS[label]
    if len(t): axes[0].plot(t*1e9,np.mean(stack,axis=0)*1e3,color=color,label=label)
    frame=result['frame']; axes[1].scatter(frame.height_V*1e3,frame.charge_C*1e12,s=4,alpha=.15,color=color,label=label)
axes[0].set(xlabel='Time relative to trigger [ns]',ylabel='Mean corrected voltage [mV]',title='900 V mean triggered waveform')
axes[1].set(xlabel='Pulse height [mV]',ylabel='Signed charge [pC]',title='Pulse charge versus height')
for ax in axes: ax.grid(alpha=.25); ax.legend(fontsize=8)

## 8. Interpretation safeguards

- Use **Auto trigger** for the ordinary terminal-noise ranking.
- Powered-run pre-trigger RMS includes the PMT/readout configuration and may differ from the empty-terminal RMS.
- All three powered datasets are analyzed identically here. The original saved `PMT_PROCESSING.ipynb` output is not used because it found zero files and its short-window exposure calculation is not comparable to timestamp-derived live time.
- The selected dark rate depends on the 3σ software threshold and the original hardware trigger. Datasheet comparison additionally requires matching temperature, threshold definition, dead time, and light-tight conditions.
- Inferred anode current is selected-event rate times mean signed selected charge; it is not a direct picoammeter measurement.

In [ ]:
auto=empty_table[(empty_table.condition=='Auto trigger')&(empty_table.waveforms>0)].sort_values('rms_median_mV')
print(f"Quietest empty auto-trigger terminal: {int(auto.iloc[0].terminal)} ({auto.iloc[0].rms_median_mV:.3f} mV median RMS)")
print(f"Noisiest empty auto-trigger terminal: {int(auto.iloc[-1].terminal)} ({auto.iloc[-1].rms_median_mV:.3f} mV median RMS)")
print()
for _,r in dark_table.iterrows():
    print(f"{r.dataset}: noise {r.median_baseline_rms_mV:.3f} mV RMS, "
          f"selected rate {r.software_selected_rate_Hz:.2f} Hz, "
          f"inferred current {r.inferred_anode_current_nA:.4g} nA")